# PathSnap — Map-Matching & Route Reconstruction: Training Notebook

**UE24CS352A Machine Learning Hackathon — Team 9**

This notebook contains the complete, reproducible training workflow:

| # | Stage | Output |
|---|-------|--------|
| 1 | Dataset inventory & EDA | `results/dataset_inventory.csv`, figures, Folium EDA map |
| 2 | Reference-label derivation (route → per-GPS-point arc) | alignment statistics |
| 3 | Candidate generation (radius / k study) | candidate miss-rate table |
| 4 | **Feature extraction & selection** | feature tables, MI scores, group ablation |
| 5 | Five models from five algorithm families, group-aware CV | `results/cv_results.csv`, `results/model_comparison.csv` |
| 6 | Model selection + **Bayesian hyperparameter optimisation** (Optuna TPE) | `results/optuna_trials.csv` |
| 7 | **HMM + Viterbi** decoding, decoder tuning, classifier-only vs HMM | `results/hmm_*.csv` |
| 8 | Folium maps (GPS, network, reference, predicted) | `maps/*.html` |
| 9 | Export of the final inference artifacts used by the API | `artifacts/` |

**Problem framing.** Every GPS observation gets a small set of *candidate* road arcs near it. A binary classifier scores each
(observation, candidate) pair with P(candidate is the true arc). The per-observation argmax is the *classifier-only* prediction;
the probabilities become the *emission* scores of an HMM whose transitions follow the road-network topology, and Viterbi
decoding returns a coherent route.

**Data-partition discipline.** `train` (80 trips) is used for fitting, cross-validation and optimisation; `student_test`
(10 trips) for development-stage comparison. **`evaluator_test` is never read in this notebook** — it is only loaded by the API
at demo time, and its `.route` labels are only read by the server-side scorer.

All feature/candidate/HMM code lives in `src/pathsnap/` so that exactly the same code runs here and inside the API
(no train/serve skew).

In [ ]:
import os, sys, json, time, warnings, platform
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import folium, optuna, sklearn, lightgbm
from IPython.display import display, HTML
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from pathsnap.io import load_trip, load_route, list_trips
from pathsnap.geo import LocalProjection, haversine_m
from pathsnap.features import FEATURES, CandidateConfig, make_context, extract_features
from pathsnap.dataset import build_partition
from pathsnap.evaluation import (candidate_metrics, top1_arcs, point_continuity, route_continuity, route_overlap)
from pathsnap.hmm import HMMConfig, reconstruct_route
from pathsnap.pipeline import PathSnapModel, decode_candidates
from pathsnap.viz import trip_map, nearby_arcs

SEED = 42
np.random.seed(SEED)
TRAIN_DIR = os.path.join(ROOT, "data", "train")
STEST_DIR = os.path.join(ROOT, "data", "student_test")
# NOTE: data/evaluator_test is intentionally never referenced in this notebook.

OUT = os.path.join(ROOT, "models")
FIG, RES, MAPS, ART, CACHE = (os.path.join(OUT, d) for d in ("figures", "results", "maps", "artifacts", "cache"))
for d in (FIG, RES, MAPS, ART, CACHE):
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
sns.set_palette("deep")
def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.show()

print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__, "| lightgbm", lightgbm.__version__, "| optuna", optuna.__version__)

## 1. Dataset inventory and exploratory analysis  *(partitions: train + student_test)*

Raw files per trip: `.track` (lon, lat, t), `.nodes` (lon, lat), `.arcs` (from, to), `.route` (ordered arc ids).
A GPS row does **not** correspond to a node/arc row — the link between them has to be computed geometrically.
We check counts, types, missing values, timestamp ordering/gaps, geographic extent, network size and connectivity, and
the structure of the reference routes.

In [ ]:
def inventory(root, split):
    rows = []
    for tid in list_trips(root):
        d = os.path.join(root, tid)
        raw = load_trip(root, tid); route = load_route(root, tid)
        tr = raw.track; lon, lat, t = tr[:, 0], tr[:, 1], tr[:, 2]
        dt = np.diff(t)
        step = haversine_m(lon[:-1], lat[:-1], lon[1:], lat[1:])
        n = len(raw.nodes); a = raw.arcs
        g = coo_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(n, n))
        ncw, labw = connected_components(g, directed=True, connection="weak")
        ncs, _ = connected_components(g, directed=True, connection="strong")
        deg = np.bincount(a.ravel(), minlength=n)
        rows.append(dict(
            split=split, trip=tid, n_points=len(tr), duration_min=(t[-1] - t[0]) / 60, length_km=step.sum() / 1000,
            dt_median=np.median(dt), dt_max=dt.max(), pct_dt_1s=100 * np.mean(np.isclose(dt, 1)), n_gaps_gt5s=int((dt > 5).sum()),
            n_nonmono_t=int((dt <= 0).sum()), n_missing=int(np.isnan(tr).sum()),
            mean_speed_kmh=3.6 * step.sum() / max(t[-1] - t[0], 1), max_step_m=step.max(),
            n_nodes=n, n_arcs=len(a), isolated_nodes=int((deg == 0).sum()), weak_components=ncw, strong_components=ncs,
            largest_wcc_pct=100 * np.bincount(labw).max() / n,
            route_len=len(route), route_continuity=np.mean(a[route[:-1], 1] == a[route[1:], 0]),
            route_revisit_pct=100 * (1 - len(np.unique(route)) / len(route)),
            lon_c=lon.mean(), lat_c=lat.mean(),
            **{f"kb_{ext}": os.path.getsize(os.path.join(d, f"{tid}.{ext}")) / 1024 for ext in ("track", "nodes", "arcs", "route")}))
    return pd.DataFrame(rows)

inv = pd.concat([inventory(TRAIN_DIR, "train"), inventory(STEST_DIR, "student_test")], ignore_index=True)
inv.to_csv(os.path.join(RES, "dataset_inventory.csv"), index=False)
summary = inv.groupby("split").agg(trips=("trip", "count"), gps_points=("n_points", "sum"), median_points=("n_points", "median"),
                                    total_km=("length_km", "sum"), nodes=("n_nodes", "sum"), arcs=("n_arcs", "sum"),
                                    route_arcs=("route_len", "sum"), missing_values=("n_missing", "sum"),
                                    non_monotonic_t=("n_nonmono_t", "sum"))
display(summary)
display(inv.drop(columns=["split", "trip"]).describe().T.round(2))

In [ ]:
print("Column types  .track: float lon, float lat, float t | .nodes: float lon, lat | .arcs: int from, to | .route: int arc_id")
print("Trips with any missing GPS value:", int((inv.n_missing > 0).sum()))
print("Trips with non-increasing timestamps:", int((inv.n_nonmono_t > 0).sum()))
print("Reference routes that are fully continuous (to(a_k) == from(a_k+1)):", int((inv.route_continuity == 1).sum()), "/", len(inv))
print("Share of 1-second steps over all trips: %.1f%%" % inv.pct_dt_1s.mean())
anom = inv[(inv.dt_max > 30) | (inv.max_step_m > 150) | (inv.route_continuity < 1)]
print("\nTrips with large gaps (>30 s), big jumps (>150 m between fixes) or discontinuous reference routes:")
display(anom[["split", "trip", "n_points", "dt_max", "n_gaps_gt5s", "max_step_m", "route_continuity", "weak_components"]].round(2))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
sns.histplot(inv, x="n_points", hue="split", bins=30, ax=ax[0, 0], log_scale=True); ax[0, 0].set_title("GPS observations per trip")
sns.histplot(inv, x="length_km", hue="split", bins=30, ax=ax[0, 1]); ax[0, 1].set_title("Trajectory length (km)")
sns.histplot(inv, x="mean_speed_kmh", hue="split", bins=30, ax=ax[0, 2]); ax[0, 2].set_title("Mean speed (km/h)")
alldt = np.concatenate([np.diff(load_trip(TRAIN_DIR, t).track[:, 2]) for t in list_trips(TRAIN_DIR)])
ax[1, 0].hist(np.clip(alldt, 0, 30), bins=60, log=True, color="C3"); ax[1, 0].set_title("Sampling interval (s, clipped at 30) – train"); ax[1, 0].set_xlabel("dt (s)")
ax[1, 1].scatter(inv.n_arcs, inv.n_points, c=(inv.split == "train").map({True: "C0", False: "C1"}), s=20)
ax[1, 1].set_xscale("log"); ax[1, 1].set_yscale("log"); ax[1, 1].set_xlabel("arcs in trip network"); ax[1, 1].set_ylabel("GPS points"); ax[1, 1].set_title("Network size vs trajectory size")
ax[1, 2].scatter(inv.lon_c, inv.lat_c, c=(inv.split == "train").map({True: "C0", False: "C1"}), s=25)
ax[1, 2].set_xlabel("longitude"); ax[1, 2].set_ylabel("latitude"); ax[1, 2].set_title("Trip locations (blue=train, orange=student_test)")
savefig("eda_overview.png")
print("dt == 1 s: %.2f%% | dt > 5 s: %d steps | max dt: %.0f s" % (100 * np.mean(np.isclose(alldt, 1)), (alldt > 5).sum(), alldt.max()))

### Geographic considerations
Trips are spread over several continents, so there is no single sensible projected CRS. Each trip is projected with a
**local equirectangular projection centred on its GPS centroid** (`pathsnap.geo.LocalProjection`). Below we check that
this projection agrees with the haversine (great-circle) distance — the error is far below GPS noise, so all later
geometry (point-to-segment distance, headings, network lengths) is done in metres.

In [ ]:
errs = []
for tid in list_trips(TRAIN_DIR):
    tr = load_trip(TRAIN_DIR, tid).track
    pr = LocalProjection.from_points(tr[:, 0], tr[:, 1])
    x, y = pr.forward(tr[:, 0], tr[:, 1])
    i = np.random.default_rng(0).integers(0, len(tr), (500, 2))
    dp = np.hypot(x[i[:, 0]] - x[i[:, 1]], y[i[:, 0]] - y[i[:, 1]])
    dh = haversine_m(tr[i[:, 0], 0], tr[i[:, 0], 1], tr[i[:, 1], 0], tr[i[:, 1], 1])
    ok = dh > 50
    errs.append(dict(trip=tid, max_abs_err_m=np.max(np.abs(dp - dh)), max_rel_err_pct=100 * np.max(np.abs(dp[ok] - dh[ok]) / dh[ok])))
errs = pd.DataFrame(errs)
print("Projection vs haversine over all 80 training trips: worst absolute error %.2f m, worst relative error %.3f%%"
      % (errs.max_abs_err_m.max(), errs.max_rel_err_pct.max()))

In [ ]:
EDA_TRIP = "00000001"
ctx_e = make_context(load_trip(TRAIN_DIR, EDA_TRIP))
ref_e = load_route(TRAIN_DIR, EDA_TRIP)
from matplotlib.collections import LineCollection
ids = nearby_arcs(ctx_e, buffer=400)
segs = lambda arcs: np.stack([ctx_e.net.A[arcs], ctx_e.net.B[arcs]], axis=1)
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].add_collection(LineCollection(segs(ids), colors="0.75", linewidths=0.6))
ax[0].add_collection(LineCollection(segs(ref_e), colors="green", linewidths=2.5))
ax[0].scatter(ctx_e.xy[:, 0], ctx_e.xy[:, 1], s=3, c="C0", zorder=3)
ax[0].set_aspect("equal"); ax[0].set_title(f"Trip {EDA_TRIP}: GPS (blue), reference route (green), network (grey)"); ax[0].set_xlabel("x (m)"); ax[0].set_ylabel("y (m)")
zoom = slice(300, 420)
x0, x1 = ctx_e.xy[zoom, 0].min() - 60, ctx_e.xy[zoom, 0].max() + 60
y0, y1 = ctx_e.xy[zoom, 1].min() - 60, ctx_e.xy[zoom, 1].max() + 60
ax[1].add_collection(LineCollection(segs(ids), colors="0.6", linewidths=1))
ax[1].add_collection(LineCollection(segs(ref_e), colors="green", linewidths=3))
ax[1].scatter(ctx_e.xy[zoom, 0], ctx_e.xy[zoom, 1], s=12, c="C0", zorder=3)
ax[1].set_xlim(x0, x1); ax[1].set_ylim(y0, y1); ax[1].set_aspect("equal"); ax[1].set_title("Zoom: GPS noise around the road")
savefig("eda_trip_network.png")

m = trip_map(ctx_e, routes=[("Reference route", ref_e, "#16a34a", 5)], title=f"EDA – train trip {EDA_TRIP}")
m.save(os.path.join(MAPS, f"eda_train_{EDA_TRIP}.html"))
m

## 2. Reference labels: which arc is each GPS point on?

The `.route` file is an ordered arc list without timestamps. Per-point labels are recovered with a **monotone dynamic-programming
alignment** (`pathsnap.labels.align_route`): each GPS point is assigned to a route position, positions may never decrease in time,
and the total point-to-arc distance is minimised. This correctly handles loops (the same street driven twice) and isolated noisy
fixes. The labels are only as good as the reference route, so we inspect the aligned distance: points that are far from every
route arc indicate reference-route limitations (detours not in the route, GPS outliers).

Building the candidate tables (section 3) also computes these labels; results are cached in `models/cache/`.

In [ ]:
CFG = CandidateConfig(radius=50.0, k=8)
t0 = time.time()
tr, trp = build_partition(TRAIN_DIR, CFG, cache=os.path.join(CACHE, "train_r50_k8.pkl"))
st, stp = build_partition(STEST_DIR, CFG, cache=os.path.join(CACHE, "stest_r50_k8.pkl"))
print(f"tables ready in {time.time()-t0:.1f}s  | train candidates {len(tr):,}  | student-test candidates {len(st):,}")

al = trp.groupby("trip").align_dist.agg(["median", lambda s: s.quantile(.95), "max"]).set_axis(["median", "p95", "max"], axis=1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.clip(trp.align_dist, 0, 100), bins=100, color="C2"); ax[0].set_yscale("log"); ax[0].set_title("Point → aligned reference arc distance (m), train")
ax[1].bar(range(len(al)), al["median"].sort_values().values, color="C2"); ax[1].set_title("Median aligned distance per training trip (sorted)"); ax[1].set_ylabel("m")
savefig("label_alignment.png")
print("Median aligned distance: %.1f m; points farther than 50 m from the reference route: %.2f%%" % (trp.align_dist.median(), 100 * (trp.align_dist > 50).mean()))
print("Trips whose reference route disagrees with the GPS (median aligned distance > 25 m):")
display(al[al["median"] > 25].round(1))

## 3. Candidate generation

Arcs are straight segments, some kilometres long, so they are **densified** (a sample point every 10 m) and indexed in a KD-tree;
exact point-to-segment distances are then computed for nearby arcs. Every point keeps its `k` nearest arcs within `radius` metres
(fallback: nearest arcs at any distance). The trade-off: a larger radius/k lowers the **candidate miss rate** (true arc absent →
unrecoverable error) but adds negatives and HMM cost. We sweep both on the training partition.

In [ ]:
sweep = []
for radius, k in [(20, 8), (30, 8), (50, 8), (75, 8), (100, 8), (50, 4), (50, 6), (50, 12)]:
    c, p = build_partition(TRAIN_DIR, CandidateConfig(radius=radius, k=k)) if (radius, k) != (50, 8) else (tr, trp)
    reliable = p.align_dist <= 50
    sweep.append(dict(radius_m=radius, k=k, cands_per_point=len(c) / len(p), miss_rate_all=1 - p.ref_in_cands.mean(),
                      miss_rate_reliable_labels=1 - p.ref_in_cands[reliable].mean(), positive_rate=c.label.mean()))
sweep = pd.DataFrame(sweep)
sweep.to_csv(os.path.join(RES, "candidate_sweep.csv"), index=False)
display(sweep.round(4))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
s1 = sweep[sweep.k == 8]
ax[0].plot(s1.radius_m, 100 * s1.miss_rate_reliable_labels, "o-", label="reliable-label points"); ax[0].plot(s1.radius_m, 100 * s1.miss_rate_all, "s--", label="all points")
ax[0].set_xlabel("radius (m), k=8"); ax[0].set_ylabel("candidate miss rate (%)"); ax[0].legend(); ax[0].set_title("Miss rate vs radius")
s2 = sweep[sweep.radius_m == 50].sort_values("k")
ax[1].plot(s2.k, 100 * s2.miss_rate_reliable_labels, "o-"); ax[1].set_xlabel("k (radius 50 m)"); ax[1].set_ylabel("miss rate (%)"); ax[1].set_title("Miss rate vs k")
savefig("candidate_sweep.png")
print("Chosen: radius = 50 m, k = 8 (the miss rate flattens beyond this; the remaining misses are mostly label-noise points far from the reference route).")

## 4. Feature extraction and selection

Code: `src/pathsnap/features.py`. Each row is one (GPS point, candidate arc) pair; **no reference information is used**.

| Group | Features | Why |
|---|---|---|
| Geometric | `dist`, `dist_rank`, `dist_delta`, `dist_ratio`, `n_cand`, `proj_frac`, `off_end` | GPS error is mostly small; distance **relative** to the closest candidate matters more than absolute distance (it adapts to local GPS quality). `off_end` catches points that only project onto an arc's extension. |
| Directional | `head_diff`, `head_cos`, `head_reliab` | Separates a street from its reverse twin and from crossing streets. Heading is computed over i±2 fixes and down-weighted when the vehicle barely moved (stationary GPS heading is noise). |
| Kinematic | `speed`, `dt_prev`, `turn_rate` | Behaviour near junctions / stops, GPS gaps. |
| Context window | `win_dist_mean`, `win_dist_max`, `win_head_diff` | Distance/heading of the neighbouring fixes i−3…i+3 to the **same** arc. A single noisy fix can be closer to a parallel road, its neighbours usually are not. |
| Topology | `arc_len`, `out_deg`, `in_deg`, `has_reverse`, `conn_prev`, `conn_next` | Whether the candidate is the same as / adjacent to the nearest arc of the previous / next fix; junction structure. |

Learned preprocessing (scalers inside model pipelines) is fitted on training folds only.

In [ ]:
key = lambda d: d["trip"].astype(str) + "_" + d["pt"].astype(str)
POINT_CAP = 800   # max sampled GPS points per training trip -> long trips do not dominate
rel = trp[trp.ref_in_cands]
samp = rel.sample(frac=1.0, random_state=SEED).groupby("trip").head(POINT_CAP)
D = tr[key(tr).isin(set(key(samp)))].reset_index(drop=True)
X, y, groups = D[FEATURES].to_numpy(np.float32), D["label"].to_numpy(), D["trip"].to_numpy()
Xst, yst = st[FEATURES].to_numpy(np.float32), st["label"].to_numpy()
print(f"training sample: {D.pt.size:,} candidate rows from {samp.shape[0]:,} GPS points in {D.trip.nunique()} trips; positive rate {y.mean():.3f}")
print(f"student-test: {len(st):,} candidate rows from {len(stp):,} points (all points, misses count as errors in Top-1)")
print("Missing values in feature matrix:", int(np.isnan(X).sum()))
display(D[FEATURES + ["label"]].describe().T.round(3))

In [ ]:
show = ["dist", "dist_delta", "dist_rank", "head_diff", "head_reliab", "win_dist_mean", "win_dist_max", "win_head_diff",
        "speed", "off_end", "conn_prev", "arc_len"]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
sub = D.sample(min(len(D), 150000), random_state=SEED)
for ax, f in zip(axes.ravel(), show):
    hi = sub[f].quantile(0.99)
    for lab, col in [(0, "C3"), (1, "C2")]:
        ax.hist(np.clip(sub.loc[sub.label == lab, f], None, hi), bins=50, alpha=0.55, density=True, color=col, label=["wrong arc", "true arc"][lab])
    ax.set_title(f)
axes[0, 0].legend()
savefig("feature_distributions.png")

plt.figure(figsize=(11, 9))
sns.heatmap(sub[FEATURES].corr(method="spearman"), cmap="vlag", center=0, square=True, cbar_kws={"shrink": .7})
plt.title("Spearman correlation between features (training sample)")
savefig("feature_correlation.png")

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi_s = D.sample(min(len(D), 120000), random_state=SEED)
mi = pd.Series(mutual_info_classif(mi_s[FEATURES], mi_s["label"], random_state=SEED,
                                   discrete_features=[f in ("dist_rank", "n_cand", "conn_prev", "conn_next", "has_reverse", "out_deg", "in_deg") for f in FEATURES]),
               index=FEATURES).sort_values(ascending=False)
mi.to_csv(os.path.join(RES, "feature_mutual_information.csv"))
plt.figure(figsize=(9, 6)); mi.iloc[::-1].plot.barh(color="C4"); plt.title("Mutual information with the label (training sample)"); plt.xlabel("MI (nats)")
savefig("feature_mutual_information.png")

**Feature-group ablation.** To justify the feature set empirically, a fixed reference model (LightGBM, default settings) is trained
with each feature group removed, using the same 3-fold group-aware CV on training trips. Metric: per-point Top-1 accuracy and
candidate macro-F1 on the held-out trips.

In [ ]:
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier

GROUPS = {
    "geometric_relative": ["dist_rank", "dist_delta", "dist_ratio", "n_cand", "proj_frac", "off_end"],
    "directional": ["head_diff", "head_cos", "head_reliab", "win_head_diff"],
    "kinematic": ["speed", "dt_prev", "turn_rate"],
    "context_window": ["win_dist_mean", "win_dist_max", "win_head_diff"],
    "topology": ["arc_len", "out_deg", "in_deg", "has_reverse", "conn_prev", "conn_next"],
}

def top1_on(df, p):
    # Top-1 accuracy over the points contained in df (rows ordered by trip, pt)
    o = np.lexsort((-p, df["pt"].to_numpy(), df["trip"].to_numpy()))
    d = df.iloc[o]
    first = ~d.duplicated(["trip", "pt"])
    return float(d.loc[first.values, "label"].mean())

def group_cv(feats, n_splits=3):
    Xf = D[feats].to_numpy(np.float32); out = []
    for trn, val in GroupKFold(n_splits=n_splits).split(Xf, y, groups):
        m = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, random_state=SEED, verbose=-1).fit(Xf[trn], y[trn])
        p = m.predict_proba(Xf[val])[:, 1]
        out.append((top1_on(D.iloc[val], p), candidate_metrics(y[val], p)["f1_macro"]))
    return np.mean(out, axis=0)

abl = [dict(setting="all features", n_features=len(FEATURES), **dict(zip(["cv_top1", "cv_f1_macro"], group_cv(FEATURES))))]
abl.append(dict(setting="distance only", n_features=1, **dict(zip(["cv_top1", "cv_f1_macro"], group_cv(["dist"])))))
for g, fs in GROUPS.items():
    keep = [f for f in FEATURES if f not in fs]
    abl.append(dict(setting=f"without {g}", n_features=len(keep), **dict(zip(["cv_top1", "cv_f1_macro"], group_cv(keep)))))
abl = pd.DataFrame(abl); abl.to_csv(os.path.join(RES, "feature_group_ablation.csv"), index=False)
display(abl.round(4))
plt.figure(figsize=(9, 4)); plt.barh(abl.setting[::-1], abl.cv_top1[::-1], color="C1"); plt.xlim(abl.cv_top1.min() - 0.02, 1)
plt.xlabel("3-fold group CV Top-1 accuracy"); plt.title("Feature-group ablation (LightGBM reference model)")
savefig("feature_ablation.png")

## 5. Model experimentation — five algorithm families

| Model | Family | Why it is a sensible candidate for PathSnap |
|---|---|---|
| Logistic Regression | Linear / probabilistic | Well-calibrated baseline; tests whether a linear score of the features is enough. |
| K-Nearest Neighbours (distance-weighted) | Instance-based | Similar (distance, heading, context) patterns should share labels; non-parametric. |
| Random Forest | Bagging ensemble | Non-linear interactions (e.g. heading only matters when the car moves), robust to scale. |
| LightGBM | Gradient boosting | Strong on tabular data, fast inference, handles interactions and monotone trends. |
| MLP (2 hidden layers) | Neural network | Smooth non-linear decision function; provides epoch-wise loss curves. |

Protocol: **5-fold GroupKFold over the 80 training trips** (a trip never appears in both train and validation). Metrics per fold:
candidate-level accuracy, macro precision/recall/F1 (threshold 0.5), log loss, and observation-level Top-1 accuracy.
Training-fold scores (on a 30k-row subsample) are recorded to diagnose over/under-fitting. Each model is then refitted on the
full training sample and evaluated on the 10 student-test trips (Top-1 over **all** points — candidate misses count as errors),
including classifier-only route continuity and inference time. KNN is fitted on a 120k-row random subsample (memory/latency).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone

KNN_MAX_ROWS = 120_000

def make_model(name, **params):
    if name == "LogReg":
        return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=3000, **({"C": 1.0} | params)))])
    if name == "KNN":
        return Pipeline([("scale", StandardScaler()), ("clf", KNeighborsClassifier(n_jobs=-1, **({"n_neighbors": 25, "weights": "distance"} | params)))])
    if name == "RandomForest":
        return RandomForestClassifier(n_jobs=-1, random_state=SEED, **({"n_estimators": 200, "min_samples_leaf": 5, "max_features": "sqrt"} | params))
    if name == "LightGBM":
        return LGBMClassifier(random_state=SEED, verbose=-1, **({"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63,
                              "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8} | params))
    if name == "MLP":
        return Pipeline([("scale", StandardScaler()), ("clf", MLPClassifier(early_stopping=True, validation_fraction=0.1, n_iter_no_change=6,
                          max_iter=80, random_state=SEED, **({"hidden_layer_sizes": (64, 32), "alpha": 1e-4, "batch_size": 1024,
                          "learning_rate_init": 1e-3} | params)))])
    raise KeyError(name)

FAMILY = {"LogReg": "Linear / probabilistic", "KNN": "Instance-based", "RandomForest": "Bagging ensemble (trees)",
          "LightGBM": "Gradient boosting", "MLP": "Neural network"}
CONFIG = {"LogReg": "L2, C=1.0, standardised", "KNN": "k=25, distance-weighted, standardised, 120k-row fit",
          "RandomForest": "200 trees, min_leaf=5, max_features=sqrt", "LightGBM": "400 trees, lr=0.05, 63 leaves, bagging 0.8",
          "MLP": "64-32 ReLU, Adam lr=1e-3, L2=1e-4, early stopping"}
MODEL_NAMES = list(FAMILY)

def fit_model(name, Xf, yf, **params):
    m = make_model(name, **params)
    if name == "KNN" and len(Xf) > KNN_MAX_ROWS:
        idx = np.random.default_rng(SEED).choice(len(Xf), KNN_MAX_ROWS, replace=False); Xf, yf = Xf[idx], yf[idx]
    t0 = time.perf_counter(); m.fit(Xf, yf)
    return m, time.perf_counter() - t0

gkf = GroupKFold(n_splits=5)
FOLDS = list(gkf.split(X, y, groups))
cv_rows = []
for name in MODEL_NAMES:
    for f, (trn, val) in enumerate(FOLDS):
        m, fit_s = fit_model(name, X[trn], y[trn])
        t0 = time.perf_counter(); p = m.predict_proba(X[val])[:, 1]; pred_s = time.perf_counter() - t0
        sub_i = np.random.default_rng(f).choice(trn, min(30000, len(trn)), replace=False)
        p_tr = m.predict_proba(X[sub_i])[:, 1]
        mv = candidate_metrics(y[val], p); mt = candidate_metrics(y[sub_i], p_tr)
        cv_rows.append(dict(model=name, fold=f, **mv, top1=top1_on(D.iloc[val], p), train_f1_macro=mt["f1_macro"],
                            train_log_loss=mt["log_loss"], fit_s=fit_s, predict_ms_per_1k=1000 * pred_s / len(val) * 1000))
    r = pd.DataFrame(cv_rows).query("model == @name")
    print(f"{name:13s} CV macro-F1 {r.f1_macro.mean():.4f} ± {r.f1_macro.std():.4f} | Top-1 {r.top1.mean():.4f} | log-loss {r.log_loss.mean():.4f} | train-F1 {r.train_f1_macro.mean():.4f} | fit {r.fit_s.mean():.1f}s")
cv = pd.DataFrame(cv_rows); cv.to_csv(os.path.join(RES, "cv_results.csv"), index=False)

In [ ]:
st_ctx = {tid: make_context(load_trip(STEST_DIR, tid)) for tid in list_trips(STEST_DIR)}
st_ref = {tid: g.ref_arc.to_numpy() for tid, g in stp.groupby("trip")}
st_rows_by_trip = {tid: g.index.to_numpy() for tid, g in st.groupby("trip")}

def student_test_eval(model, name):
    t0 = time.perf_counter(); p = model.predict_proba(Xst)[:, 1]; inf_s = time.perf_counter() - t0
    m = candidate_metrics(yst, p)
    correct = n_pts = 0; conts = []
    preds = {}
    for tid, rows in st_rows_by_trip.items():
        g = st.loc[rows]
        arcs = top1_arcs(g.pt.to_numpy(), g.arc.to_numpy(), p[rows], len(st_ref[tid]))
        preds[tid] = arcs
        correct += (arcs == st_ref[tid]).sum(); n_pts += len(arcs)
        conts.append(point_continuity(st_ctx[tid].net, arcs))
    return dict(model=name, st_top1=correct / n_pts, st_accuracy=m["accuracy"], st_precision_macro=m["precision_macro"],
                st_recall_macro=m["recall_macro"], st_f1_macro=m["f1_macro"], st_log_loss=m["log_loss"],
                st_route_continuity=float(np.mean(conts)), inference_ms_per_1k=1000 * inf_s / len(yst) * 1000), p, preds

final_models, st_rows, st_probs, st_preds, fit_times = {}, [], {}, {}, {}
for name in MODEL_NAMES:
    final_models[name], fit_times[name] = fit_model(name, X, y)
    row, st_probs[name], st_preds[name] = student_test_eval(final_models[name], name)
    row["train_time_s"] = fit_times[name]; st_rows.append(row)
    print(f"{name:13s} student-test Top-1 {row['st_top1']:.4f} | macro-F1 {row['st_f1_macro']:.4f} | log-loss {row['st_log_loss']:.4f} | continuity {row['st_route_continuity']:.3f} | {row['inference_ms_per_1k']:.1f} ms/1k")
st_res = pd.DataFrame(st_rows)

In [ ]:
agg = cv.groupby("model").agg(cv_f1_macro=("f1_macro", "mean"), cv_f1_std=("f1_macro", "std"), cv_top1=("top1", "mean"),
                              cv_log_loss=("log_loss", "mean"), cv_precision=("precision_macro", "mean"), cv_recall=("recall_macro", "mean"),
                              train_f1_macro=("train_f1_macro", "mean")).reset_index()
comp = agg.merge(st_res, on="model")
comp.insert(1, "family", comp.model.map(FAMILY)); comp.insert(2, "key_configuration", comp.model.map(CONFIG))
comp["overfit_gap_f1"] = comp.train_f1_macro - comp.cv_f1_macro
comp = comp.sort_values("cv_f1_macro", ascending=False).reset_index(drop=True)
comp.to_csv(os.path.join(RES, "model_comparison.csv"), index=False)
display(comp[["model", "family", "key_configuration", "cv_f1_macro", "cv_f1_std", "cv_top1", "st_top1", "st_f1_macro",
              "st_log_loss", "st_route_continuity", "inference_ms_per_1k", "train_time_s", "overfit_gap_f1"]].round(4))

In [ ]:
order = comp.model.tolist()
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
sns.boxplot(data=cv, x="model", y="f1_macro", order=order, ax=ax[0, 0], color="C0"); sns.stripplot(data=cv, x="model", y="f1_macro", order=order, ax=ax[0, 0], color="k", size=4)
ax[0, 0].set_title("5-fold group CV macro-F1 (per fold)")
w = 0.38; xx = np.arange(len(order)); c = comp.set_index("model").loc[order]
ax[0, 1].bar(xx - w / 2, c.st_top1, w, label="Top-1 accuracy"); ax[0, 1].bar(xx + w / 2, c.st_f1_macro, w, label="macro-F1")
ax[0, 1].set_xticks(xx, order); ax[0, 1].set_ylim(min(c.st_top1.min(), c.st_f1_macro.min()) - 0.05, 1); ax[0, 1].legend(); ax[0, 1].set_title("Student-test performance")
ax[1, 0].bar(xx - w / 2, c.cv_log_loss, w, label="CV"); ax[1, 0].bar(xx + w / 2, c.st_log_loss, w, label="student-test"); ax[1, 0].set_xticks(xx, order); ax[1, 0].legend(); ax[1, 0].set_title("Log loss (lower is better)")
ax[1, 1].bar(xx, c.inference_ms_per_1k, color="C5"); ax[1, 1].set_xticks(xx, order); ax[1, 1].set_yscale("log"); ax[1, 1].set_title("Inference time (ms per 1,000 candidates, log scale)")
savefig("model_comparison.png")

from sklearn.metrics import confusion_matrix
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
for a, name in zip(ax, order):
    cm = confusion_matrix(yst, (st_probs[name] >= 0.5).astype(int), normalize="true")
    sns.heatmap(cm, annot=True, fmt=".3f", cmap="Blues", cbar=False, ax=a, xticklabels=["wrong", "true"], yticklabels=["wrong", "true"])
    a.set_title(name); a.set_xlabel("predicted"); a.set_ylabel("actual")
plt.suptitle("Student-test candidate confusion matrices (row-normalised)")
savefig("confusion_matrices.png")

**Training histories.** Only the MLP (epochs) and LightGBM (boosting iterations) produce training histories. Logistic Regression,
KNN and Random Forest do not, so no curves are drawn for them. LightGBM's curve uses a group-held-out 20 % of training trips.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
mlp = final_models["MLP"].named_steps["clf"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(mlp.loss_curve_, label="training loss (log loss + L2)")
ax2 = ax[0].twinx(); ax2.plot(mlp.validation_scores_, color="C1", label="validation accuracy"); ax2.set_ylabel("validation accuracy")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss"); ax[0].set_title(f"MLP training history ({mlp.n_iter_} epochs, early stopping)")
ax[0].legend(loc="upper center"); ax2.legend(loc="center right")
trn, val = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, groups))
lg = make_model("LightGBM").fit(X[trn], y[trn], eval_set=[(X[trn], y[trn]), (X[val], y[val])], eval_names=["train", "valid"], eval_metric="binary_logloss")
for k, v in lg.evals_result_.items():
    ax[1].plot(v["binary_logloss"], label=k)
ax[1].set_xlabel("boosting iteration"); ax[1].set_ylabel("log loss"); ax[1].set_title("LightGBM train vs validation log loss"); ax[1].legend()
savefig("training_histories.png")

### Model selection

Primary selection metric: **5-fold group-CV macro-F1** (balanced over the true-arc and wrong-arc classes, computed on unseen trips,
and the metric requested in the comparison table). Tie-breakers / sanity checks: CV Top-1 accuracy (directly the map-matching
objective), log loss (probability quality matters because the probabilities are the HMM emissions), fold stability, and inference time.
Student-test results are reported for comparison but are *not* used as the primary criterion.

In [ ]:
best = comp.iloc[0]
BEST = best.model
print(f"Selected model: {BEST}  (CV macro-F1 {best.cv_f1_macro:.4f} ± {best.cv_f1_std:.4f}, CV Top-1 {best.cv_top1:.4f}, CV log-loss {best.cv_log_loss:.4f})")
print("Ranking by CV macro-F1:", " > ".join(comp.model))
print("Overfitting check (train-fold F1 − CV F1):", dict(zip(comp.model, comp.overfit_gap_f1.round(4))))

## 6. Bayesian hyperparameter optimisation (Optuna, TPE sampler)

* **Search**: Tree-structured Parzen Estimator (sequential model-based Bayesian optimisation), seeded for reproducibility.
* **Objective**: mean macro-F1 over a **3-fold GroupKFold** on the 80 training trips (student-test and evaluator data are never used).
* **Trials**: 30. Every trial is stored in `results/optuna_trials.csv` (and the Optuna journal file).
* Search spaces are defined for all five families; the one for the selected model is used.

In [ ]:
SPACES = {
    "LightGBM": lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 800, step=50),
                               learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                               num_leaves=t.suggest_int("num_leaves", 15, 255, log=True),
                               min_child_samples=t.suggest_int("min_child_samples", 5, 200, log=True),
                               subsample=t.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
                               colsample_bytree=t.suggest_float("colsample_bytree", 0.4, 1.0),
                               reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10, log=True),
                               reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10, log=True)),
    "RandomForest": lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 400, step=50),
                                   max_depth=t.suggest_int("max_depth", 6, 40),
                                   min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 50, log=True),
                                   max_features=t.suggest_categorical("max_features", ["sqrt", "log2", 0.5])),
    "LogReg": lambda t: dict(C=t.suggest_float("C", 1e-3, 1e2, log=True)),
    "KNN": lambda t: dict(n_neighbors=t.suggest_int("n_neighbors", 5, 100, log=True),
                          weights=t.suggest_categorical("weights", ["uniform", "distance"]), p=t.suggest_categorical("p", [1, 2])),
    "MLP": lambda t: dict(hidden_layer_sizes=tuple(int(v) for v in t.suggest_categorical("hidden_layers", ["32", "64-32", "128-64", "128-64-32"]).split("-")),
                          alpha=t.suggest_float("alpha", 1e-6, 1e-2, log=True),
                          learning_rate_init=t.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
                          batch_size=t.suggest_categorical("batch_size", [256, 512, 1024])),
}
HPO_FOLDS = list(GroupKFold(n_splits=3).split(X, y, groups))

def objective(trial):
    params = SPACES[BEST](trial)
    scores, lls = [], []
    for trn, val in HPO_FOLDS:
        m, _ = fit_model(BEST, X[trn], y[trn], **params)
        p = m.predict_proba(X[val])[:, 1]; mv = candidate_metrics(y[val], p)
        scores.append(mv["f1_macro"]); lls.append(mv["log_loss"])
    trial.set_user_attr("cv_log_loss", float(np.mean(lls)))
    return float(np.mean(scores))

optuna.logging.set_verbosity(optuna.logging.WARNING)
try:
    from optuna.storages import JournalStorage
    from optuna.storages.journal import JournalFileBackend, JournalFileOpenLock
    jpath = os.path.join(RES, "optuna_journal.log")
    for f_ in (jpath, jpath + ".lock"):
        if os.path.exists(f_): os.remove(f_)
    storage = JournalStorage(JournalFileBackend(jpath, lock_obj=JournalFileOpenLock(jpath)))  # open-lock works on Windows without symlink rights
    storage.get_all_studies()
except Exception as e:
    print("journal storage unavailable, using in-memory:", e); storage = None
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED), study_name=f"pathsnap_{BEST}", storage=storage)
t0 = time.time()
study.optimize(objective, n_trials=30, show_progress_bar=False)
hpo_cost = time.time() - t0
trials = study.trials_dataframe(); trials.to_csv(os.path.join(RES, "optuna_trials.csv"), index=False)
print(f"{len(trials)} trials in {hpo_cost/60:.1f} min | best CV macro-F1 {study.best_value:.4f}")
print("Best parameters:", study.best_params)
display(trials.sort_values("value", ascending=False).head(8))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
v = trials["value"].to_numpy()
ax[0].plot(v, "o", alpha=.6, label="trial"); ax[0].plot(np.maximum.accumulate(v), "r-", label="best so far")
ax[0].set_xlabel("trial"); ax[0].set_ylabel("3-fold group CV macro-F1"); ax[0].set_title(f"Optuna TPE optimisation history ({BEST})"); ax[0].legend()
try:
    imp = optuna.importance.get_param_importances(study)
    ax[1].barh(list(imp)[::-1], list(imp.values())[::-1], color="C2"); ax[1].set_title("Hyperparameter importance (fANOVA)")
except Exception as e:
    ax[1].text(0.1, 0.5, f"importance unavailable: {e}")
savefig("optuna_history.png")

In [ ]:
# Tuned vs untuned — same data partitions and metrics
tuned_params = SPACES[BEST](optuna.trial.FixedTrial(study.best_params))   # map trial values back to estimator kwargs
rows = []
for f, (trn, val) in enumerate(FOLDS):
    m, _ = fit_model(BEST, X[trn], y[trn], **tuned_params)
    p = m.predict_proba(X[val])[:, 1]
    rows.append(dict(**candidate_metrics(y[val], p), top1=top1_on(D.iloc[val], p)))
tcv = pd.DataFrame(rows)
tuned_model, tuned_fit = fit_model(BEST, X, y, **tuned_params)
trow, tuned_p, tuned_preds = student_test_eval(tuned_model, BEST + " (tuned)")
u = comp.set_index("model").loc[BEST]
tvu = pd.DataFrame({
    "untuned": [u.cv_f1_macro, u.cv_top1, u.cv_log_loss, u.st_top1, u.st_f1_macro, u.st_log_loss, u.inference_ms_per_1k, u.train_time_s],
    "tuned":   [tcv.f1_macro.mean(), tcv.top1.mean(), tcv.log_loss.mean(), trow["st_top1"], trow["st_f1_macro"], trow["st_log_loss"], trow["inference_ms_per_1k"], tuned_fit],
}, index=["CV macro-F1 (5-fold)", "CV Top-1", "CV log loss", "Student-test Top-1", "Student-test macro-F1", "Student-test log loss", "Inference ms / 1k cand.", "Train time (s)"])
tvu.to_csv(os.path.join(RES, "tuned_vs_untuned.csv"))
display(tvu.round(4))
USE_TUNED = tcv.f1_macro.mean() >= u.cv_f1_macro
FINAL_MODEL = tuned_model if USE_TUNED else final_models[BEST]
FINAL_PARAMS = tuned_params if USE_TUNED else {}
final_p_st = tuned_p if USE_TUNED else st_probs[BEST]
print("Final classifier:", BEST, "(tuned)" if USE_TUNED else "(untuned: tuning did not improve CV macro-F1)")
fig, ax = plt.subplots(figsize=(8, 4))
sel = tvu.loc[["CV macro-F1 (5-fold)", "CV Top-1", "Student-test Top-1", "Student-test macro-F1"]]
sel.plot.bar(ax=ax, rot=15); ax.set_ylim(sel.values.min() - 0.03, 1); ax.set_title(f"{BEST}: untuned vs Bayesian-tuned")
savefig("tuned_vs_untuned.png")

In [ ]:
from sklearn.inspection import permutation_importance
idx = np.random.default_rng(SEED).choice(len(Xst), min(30000, len(Xst)), replace=False)
pi = permutation_importance(FINAL_MODEL, Xst[idx], yst[idx], scoring="neg_log_loss", n_repeats=3, random_state=SEED, n_jobs=1)
pim = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
pim.to_csv(os.path.join(RES, "final_model_permutation_importance.csv"))
plt.figure(figsize=(9, 6)); pim.plot.barh(color="C0"); plt.xlabel("increase in log loss when the feature is shuffled")
plt.title(f"Permutation importance of the final {BEST} model (student-test sample)")
savefig("final_permutation_importance.png")

## 7. HMM and Viterbi route reconstruction

Implementation: `src/pathsnap/hmm.py`.

* **Hidden states** — at GPS step *i*, one state per candidate arc in *C<sub>i</sub>* (≤ 8), each keeping its arc id and the
  projection of the fix onto the arc, so the candidate ↔ road-segment correspondence is preserved.
* **Emission** — `alpha · log(p + eps)` where *p* is the final classifier's probability.
* **Transition** — Newson & Krumm style: `−|d_route − d_gps| / beta_eff`, where `d_route` is the network distance between the
  two projected positions (same arc: along-arc offset, backwards motion penalised ×3; different arcs: rest of arc + bounded Dijkstra
  shortest path + offset on the new arc) and `d_gps` the straight-line distance between the fixes. `beta_eff = beta·√dt` widens
  the tolerance during GPS gaps.
* **Unreachable / disconnected transitions** get a constant log-probability (−40) rather than −∞, so the chain never breaks; the
  decoder only "jumps" when the emissions overwhelmingly demand it. Points with no candidate within 50 m receive fallback
  candidates (nearest arcs), so there are never empty state sets.
* **Decoding** — standard Viterbi recurrence `δ_i(b) = max_a [δ_{i−1}(a) + log T(a→b)] + log E_i(b)` with back-pointers;
  the per-point arc sequence is then turned into a continuous route by inserting shortest paths between non-adjacent arcs.

**Decoder tuning** (`alpha`, `beta`) uses only training trips, with **out-of-fold** classifier probabilities (so the emissions
are as noisy as on unseen trips).

In [ ]:
# Out-of-fold probabilities for every candidate of every training trip
oof = np.zeros(len(tr))
trip_arr = tr["trip"].to_numpy()
trips_all = np.array(list_trips(TRAIN_DIR))
for trn_t, val_t in GroupKFold(n_splits=5).split(trips_all, groups=trips_all):
    tr_mask = np.isin(D["trip"].to_numpy(), trips_all[trn_t])
    m, _ = fit_model(BEST, X[tr_mask], y[tr_mask], **FINAL_PARAMS)
    vm = np.isin(trip_arr, trips_all[val_t])
    oof[vm] = m.predict_proba(tr.loc[vm, FEATURES].to_numpy(np.float32))[:, 1]

tune_trips = list(pd.Series(trips_all).sample(24, random_state=SEED))
tr_ctx = {tid: make_context(load_trip(TRAIN_DIR, tid)) for tid in tune_trips}
tr_rows = {tid: np.flatnonzero(trip_arr == tid) for tid in tune_trips}
tr_ref = {tid: g.ref_arc.to_numpy() for tid, g in trp[trp.trip.isin(tune_trips)].groupby("trip")}

grid = []
t0 = time.time()
for alpha in (0.5, 1.0, 2.0):
    for beta in (2.0, 4.0, 8.0, 16.0):
        hc = HMMConfig(alpha=alpha, beta=beta)
        corr = n = 0; conts = []
        for tid in tune_trips:
            g = tr.iloc[tr_rows[tid]]
            arcs, _ = decode_candidates(tr_ctx[tid], g, oof[tr_rows[tid]], hc)
            corr += (arcs == tr_ref[tid]).sum(); n += len(arcs); conts.append(point_continuity(tr_ctx[tid].net, arcs))
        grid.append(dict(alpha=alpha, beta=beta, hmm_top1=corr / n, continuity=np.mean(conts)))
grid = pd.DataFrame(grid); grid.to_csv(os.path.join(RES, "hmm_decoder_grid.csv"), index=False)
cls_base = sum((top1_arcs(tr.iloc[tr_rows[t]].pt.to_numpy(), tr.iloc[tr_rows[t]].arc.to_numpy(), oof[tr_rows[t]], len(tr_ref[t])) == tr_ref[t]).sum() for t in tune_trips) / sum(len(tr_ref[t]) for t in tune_trips)
print(f"decoder grid on {len(tune_trips)} training trips in {time.time()-t0:.0f}s | classifier-only OOF Top-1 on the same trips: {cls_base:.4f}")
display(grid.pivot(index="alpha", columns="beta", values="hmm_top1").round(4))
bestg = grid.sort_values(["hmm_top1", "continuity"], ascending=False).iloc[0]
HMM_CFG = HMMConfig(alpha=float(bestg.alpha), beta=float(bestg.beta))
print("Selected decoder parameters:", HMM_CFG)
plt.figure(figsize=(6, 4)); sns.heatmap(grid.pivot(index="alpha", columns="beta", values="hmm_top1"), annot=True, fmt=".4f", cmap="Greens")
plt.title("HMM Top-1 on training trips (OOF emissions)"); savefig("hmm_grid.png")

In [ ]:
# Classifier-only vs classifier + HMM on the 10 student-test trips (final model, tuned decoder)
rows = []; st_hmm = {}
for tid in list_trips(STEST_DIR):
    ctx = st_ctx[tid]; ridx = st_rows_by_trip[tid]; g = st.loc[ridx]; p = final_p_st[ridx]; ref = st_ref[tid]
    rr = load_route(STEST_DIR, tid)   # reference route: evaluation only
    cls = top1_arcs(g.pt.to_numpy(), g.arc.to_numpy(), p, len(ref))
    t0 = time.perf_counter(); hmm, _ = decode_candidates(ctx, g, p, HMM_CFG); dec_s = time.perf_counter() - t0
    route_c, _, _ = reconstruct_route(ctx.net, cls); route_h, filled, unfilled = reconstruct_route(ctx.net, hmm)
    st_hmm[tid] = (cls, hmm, route_h)
    oc, oh = route_overlap(ctx.net, route_c, rr), route_overlap(ctx.net, route_h, rr)
    rows.append(dict(trip=tid, n_points=len(ref), candidate_miss=1 - stp[stp.trip == tid].ref_in_cands.mean(),
                     top1_classifier=(cls == ref).mean(), top1_hmm=(hmm == ref).mean(),
                     point_continuity_classifier=point_continuity(ctx.net, cls), point_continuity_hmm=point_continuity(ctx.net, hmm),
                     route_continuity_hmm=route_continuity(ctx.net, route_h),
                     route_f1_classifier=oc["route_f1"], route_f1_hmm=oh["route_f1"],
                     route_precision_hmm=oh["route_precision"], route_recall_hmm=oh["route_recall"], decode_s=dec_s))
hc = pd.DataFrame(rows)
tot = hc.select_dtypes("number").mean(); tot["n_points"] = hc.n_points.sum()
tot["top1_classifier"] = np.average(hc.top1_classifier, weights=hc.n_points); tot["top1_hmm"] = np.average(hc.top1_hmm, weights=hc.n_points)
hc = pd.concat([hc, pd.DataFrame([dict(trip="ALL (point-weighted)", **tot)])], ignore_index=True)
hc.to_csv(os.path.join(RES, "hmm_vs_classifier_student_test.csv"), index=False)
display(hc.round(4))

In [ ]:
h = hc.iloc[:-1]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5)); xx = np.arange(len(h)); w = 0.4
ax[0].bar(xx - w/2, h.top1_classifier, w, label="classifier only"); ax[0].bar(xx + w/2, h.top1_hmm, w, label="classifier + HMM")
ax[0].set_xticks(xx, h.trip, rotation=45); ax[0].set_ylim(h.top1_classifier.min() - .05, 1.0); ax[0].set_title("Observation-level Top-1 accuracy"); ax[0].legend()
ax[1].bar(xx - w/2, h.point_continuity_classifier, w); ax[1].bar(xx + w/2, h.point_continuity_hmm, w)
ax[1].set_xticks(xx, h.trip, rotation=45); ax[1].set_title("Continuity of consecutive matches")
ax[2].bar(xx - w/2, h.route_f1_classifier, w); ax[2].bar(xx + w/2, h.route_f1_hmm, w)
ax[2].set_xticks(xx, h.trip, rotation=45); ax[2].set_ylim(h.route_f1_classifier.min() - .05, 1.0); ax[2].set_title("Route-level F1 (length-weighted arc overlap)")
savefig("hmm_vs_classifier.png")
a = hc.iloc[-1]
print(f"Overall Top-1: classifier {a.top1_classifier:.4f} -> HMM {a.top1_hmm:.4f}; continuity {a.point_continuity_classifier:.3f} -> {a.point_continuity_hmm:.3f}; route F1 {a.route_f1_classifier:.4f} -> {a.route_f1_hmm:.4f}")
worse = h[h.top1_hmm < h.top1_classifier]
print("Trips where HMM decoding did not help:", "none" if worse.empty else ", ".join(worse.trip))

## 8. Geographic visualisation (Folium)

For every student-test trip an interactive map is written to `models/maps/` with toggleable layers: raw GPS, nearby road
network, reference route (green), HMM-predicted route (red), and the GPS fixes where the classifier-only prediction was wrong but
HMM fixed it (orange). The worst and best trips are shown inline.

In [ ]:
for tid in list_trips(STEST_DIR):
    ctx = st_ctx[tid]; cls, hmm, route_h = st_hmm[tid]; ref = st_ref[tid]
    fixed = np.flatnonzero((cls != ref) & (hmm == ref))
    still = np.flatnonzero(hmm != ref); still = still[::max(1, len(still) // 1500)]
    m = trip_map(ctx, routes=[("Reference route", load_route(STEST_DIR, tid), "#16a34a", 7), ("Predicted route (HMM)", route_h, "#dc2626", 3)],
                 points=[("Fixed by HMM", ctx.track[fixed, 1], ctx.track[fixed, 0], "#f59e0b"),
                         ("Still mismatched", ctx.track[still, 1], ctx.track[still, 0], "#7c3aed")],
                 title=f"student_test {tid} — Top-1 {np.mean(hmm == ref):.3f}")
    m.save(os.path.join(MAPS, f"student_test_{tid}.html"))
worst = h.sort_values("top1_hmm").trip.iloc[0]
print("maps written to", MAPS, "| showing worst trip", worst)
display(HTML(f'<iframe src="maps/student_test_{worst}.html" width="100%" height="520" style="border:0"></iframe>'))

## 9. Export inference artifacts

`artifacts/final_model.joblib` (classifier incl. any scaler) and `artifacts/pipeline_config.json` (feature order, candidate
configuration, tuned HMM parameters, provenance). The API loads exactly these files. A reload check runs the complete pipeline on
one student-test trip.

In [ ]:
meta = dict(selected_model=BEST, tuned=bool(USE_TUNED), params=FINAL_PARAMS, seed=SEED, trained_on="train (80 trips)",
            n_training_rows=int(len(D)), point_cap_per_trip=POINT_CAP,
            cv_f1_macro=float(tcv.f1_macro.mean() if USE_TUNED else u.cv_f1_macro),
            student_test_top1_classifier=float(hc.iloc[-1].top1_classifier), student_test_top1_hmm=float(hc.iloc[-1].top1_hmm),
            created=time.strftime("%Y-%m-%d %H:%M:%S"))
PathSnapModel(FINAL_MODEL, FEATURES, CFG, HMM_CFG, meta).save(ART)
comp.to_csv(os.path.join(ART, "model_comparison.csv"), index=False)

reloaded = PathSnapModel.load(ART)
tid = list_trips(STEST_DIR)[0]
res = reloaded.predict_trip(load_trip(STEST_DIR, tid))
print("reloaded pipeline on", tid, "| Top-1 (HMM) = %.4f" % np.mean(res.hmm_arcs == st_ref[tid]), "| timings:", {k: round(v, 3) for k, v in res.timings.items()})
print(json.dumps(meta, indent=2, default=str))

## 10. Summary

The tables above (`results/model_comparison.csv`, `results/tuned_vs_untuned.csv`, `results/hmm_vs_classifier_student_test.csv`)
contain every number needed for the report. Key observations to discuss in the viva:

* **Features** — relative distance, heading agreement and the context-window distances carry most of the signal (see MI, ablation
  and permutation-importance plots); distance alone is clearly insufficient, especially on parallel roads and two-way streets.
* **Models** — compare the five families on CV macro-F1, stability across folds, log loss (probability quality for the HMM),
  and inference speed; the train-vs-CV gap column diagnoses over-fitting.
* **HMM** — improves observation-level Top-1 and strongly improves continuity, because it rejects isolated jumps to unconnected
  arcs; residual errors come from candidate misses (reference arc not within 50 m — often reference-route limitations) and
  very short arcs at junctions.
* **Limitations** — labels are derived from a reference route that is not independently verified; trips whose GPS deviates from
  their reference route (see section 2) cap the achievable accuracy.